# OpenAI Agents SDK

## Asyncio

* It is light weight of mutli-treading

All your methods and functions start with `async`
Anytime you call them, use `await`

* Functions defined with `async def` are called **coroutines** --they are special functions that can be paused and resumed

* Calling a coroutine does not execute it immediately -- it returns a coroutine object

* To actually run a coroutine, you must await it, which schedules it for execution within an event loop

* While a coroutine is waiting (e.g. for I/O), the event loop can run other coroutines


## OpendAI Agent SDK
- lightweight and flexible
- Stays out of the way
- Makes common activities easy

### In OpenAI
- Agents represent LLMs
- Handoffs represent interactions
- Gaurdrails represent controls

### Three steps
1. Create an instance of Agent
2. Use with `trace()` to track the agent
3. Call `runner.run()` to run the agent

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
load_dotenv(override=True)

True

In [2]:
agent = Agent(name="Jockester", instructions="You are ajoke teller", model="gpt-4o-mini")

* instruction is the system prompt which specifies the task for the agent

In [8]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell me a joke about Autonomous AI agents")
    print(result.final_output)

Why did the autonomous AI agent bring a ladder to its job?

Because it wanted to reach new heights in decision-making!


Tips for successful vibing

* *Good vibes*: prompt well - ask for short answers and latest APIs for today's date
* *Vibe but verify*: ask 2 LLMs the same question
* *Step up the vibe*: ask to break down your request into independently testable steps
* *Vibe and validate*: ask an LLM then get another LLM to check
* *Vibe with variety*: ask for 3 solutions to the same problem, pick the best

# Day 2

In [2]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [4]:
load_dotenv(override=True)

True

In [5]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [6]:
sales_agent1 = Agent(
    name="Professional Sales Agent", instructions=instructions1, model="gpt-4o-mini")

sales_agent2 = Agent(
    name="Professional Sales Agent", instructions=instructions2, model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Professional Sales Agent", instructions=instructions3, model="gpt-4o-mini"
)

* For streaming the results use `run_stream` instead of `run` 
    * The you can loop over the streaming using `result.stream_events()`

In [7]:
result = Runner.run_streamed(sales_agent1, "Write a cold sales email")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC 2 Compliance with ComplAI

Hi [Recipient's Name],

I hope this message finds you well. I’m [Your Name], reaching out from ComplAI, where we specialize in providing AI-driven solutions for SOC 2 compliance.

In today’s data-driven landscape, ensuring compliance and preparing for audits can be a daunting task. Our platform automates the compliance process, streamlines documentation, and helps mitigate risks, ultimately saving you time and reducing costs.

Companies like [Example Company] have transformed their compliance processes with ComplAI, achieving faster audit preparation and enhancing their security posture.

I would love the opportunity to discuss how ComplAI can support your organization's compliance efforts and help prepare you for upcoming audits. Are you available for a brief call next week? 

Thank you for your time, and I look forward to the possibility of working together.

Best regards,

[Your Name]  
[Your Title]  
ComplAI  
[Your Phone Numb

* To run different async in parallel use `asyncio.gather`. This gather and keep the results

In [8]:
message = "write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Streamline Your SOC 2 Compliance with ComplAI

Dear [Recipient's Name],

I hope this message finds you well. I am [Your Name], with ComplAI, where we specialize in enhancing compliance processes through advanced AI technology.

Navigating SOC 2 compliance can be complex and time-consuming. Our SaaS tool is designed to simplify and streamline this journey, enabling organizations like yours to efficiently prepare for audits while mitigating risks and ensuring data security.

With ComplAI, you can expect:
- Automated documentation and evidence collection
- Real-time compliance tracking
- Reduced audit preparation time

I would love the opportunity to discuss how ComplAI can support your compliance efforts and empower your team to focus on core business objectives. Are you available for a brief call this week?

Thank you for considering this solution for your compliance needs. I look forward to the possibility of working together.

Best regards,

[Your Name]  
[Your Position]  
Co

In [9]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
            Imagine you are a customer and pick the one you are most likely to respond to. \
            Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini",
)

In [10]:
emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

best = await Runner.run(sales_picker, emails)

print(f"Best sales email:\n{best.final_output}")

Best sales email:
Subject: Is Your SOC2 Compliance Playing Hide and Seek? 🤔✨

Hi [Recipient's Name],

I hope this email finds you well and not buried under a mountain of compliance paperwork! If you are, I’ve got good news: ComplAI is here to rescue you faster than a superhero with a spreadsheet!

Imagine a world where audits don’t feel like a root canal—painful and long. With ComplAI, you can say goodbye to the stress of SOC2 compliance. Our AI-powered tool ensures that all your ducks are in a row, so you can focus on what really matters: your business and maybe even enjoying that extra slice of pizza for lunch! 🍕 

Here’s the kicker: we make compliance as easy as clicking “snooze” on your alarm (seriously, who doesn’t love that?). 

Want to chat about how we can turn your audit journey into a walk in the park (the kind with ice cream, of course)? Let’s hop on a quick call!

Looking forward to hearing from you!

Best,  
[Your Name]  
[Your Title]  
ComplAI  
P.S. If you’d prefer an em

* Using decorator `@function_tool` we can change a function into a tool

In [31]:
@function_tool
def send_email(body: str) -> Dict[str, str]:
    """Send a plain-text email whose first line can optionally be `Subject: ...`."""
    cleaned_body = body.strip()
    if not cleaned_body:
        raise ValueError("Email body must not be empty")

    subject = "Cold sales email from ComplAI"
    message_body = cleaned_body
    first_line, separator, remainder = cleaned_body.partition("\n")
    if first_line.lower().startswith("subject:"):
        parsed_subject = first_line.split(":", 1)[1].strip()
        if parsed_subject:
            subject = parsed_subject
        message_body = remainder.lstrip() if separator else ""

    if not message_body:
        raise ValueError("Email message body must not be empty")

    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("h.emamgholizadeh@gmail.com")  # Change to your verified sender
    to_email = To("h_emamgholizadeh@yahoo.com")  # Change to your recipient
    content = Content("text/plain", message_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [32]:
send_email

FunctionTool(name='send_email', description='Send a plain-text email whose first line can optionally be `Subject: ...`.', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x103f10040>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

* You can convert an agent to a tool using `agent.as_tool`

In [33]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [34]:
tools = [tool1, tool2, tool3, send_email]
tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10d0419e0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10d040c20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', description='Write 

* Add tools to an agent using `tool=` parameter in the agent

In [35]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool exactly once with the selected draft. The draft should include a `Subject:` line followed by the email body.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
- Do not retry send_email if it fails. Report the failure instead.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")

message = "Send a cold sales email address to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)



In [26]:
result

RunResult(input="Send a cold sales email address to 'Dear CEO'", new_items=[ToolCallItem(agent=Agent(name='Sales Manager', handoff_description=None, tools=[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10b33ff60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10b33cea0>, 

### Handoffs
* Handoffs represent a way an agent can delegate to an agent, passing control to it
    * Handoffs and Agents-as-tools are similar
        * In both cases, an Agent can collaborate with another Agent
        * With tools, control passes back
        * With handoffs, control passes across

In [36]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [37]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send out an email with the given subject and HTML body to all sales prospects"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("h.emamgholizadeh@gmail.com")
    to_email = To("h_emamgholizadeh@yahoo.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [38]:
tools = [subject_tool, html_tool, send_html_email]

In [39]:
instructions = "You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it",
)

* You define the handoffs as a new list to the `handoffs` parameter 

In [40]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10d0419e0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10d040c20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent3', description='Write a 

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini",
)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)